In [ ]:
using LinearAlgebra, Printf
DEV = true
if DEV
    include("../src/EDKit.jl")
    using .EDKit
else
    using EDKit
end
using ITensors, ITensorMPS


# fDAOE example

This notebook shows the fermionic variant `fdaoe(ps, l, gamma)`. Its weighting is more structure-sensitive than the bosonic `daoe`, so the safest way to understand the implemented rule is to inspect a few short strings and then compare both truncations inside the same operator-growth run.


In [ ]:
pad_labels(labels, L) = vcat(labels, fill("I", L - length(labels)))

function pauli_string_mps(ps, labels)
    productMPS(ps, pad_labels(labels, length(ps)))
end

function normalized_weight(D, psi)
    numer = real(inner(psi, apply(D, psi)))
    denom = real(inner(psi, psi))
    numer / denom
end


## Compare DAOE and fDAOE on short strings

The package tests already confirm a few benchmark cases such as `XX`, `ZZ`, and `XZ`. The table below makes the implementation-specific distinction explicit for a broader but still small set of strings.


In [ ]:
L = 6
ps = siteinds("Pauli", L)
gamma = 0.3
D = daoe(ps, 2, gamma)
FD = fdaoe(ps, 2, gamma)

samples = [
    ["X"],
    ["Z"],
    ["X", "X"],
    ["Z", "Z"],
    ["X", "Z"],
    ["X", "X", "X"],
    ["X", "Z", "X"],
]

[(;
    string = join(labels),
    daoe = normalized_weight(D, pauli_string_mps(ps, labels)),
    fdaoe = normalized_weight(FD, pauli_string_mps(ps, labels)),
) for labels in samples]


## Compare the two truncations during operator growth

We evolve a mixed string so that both truncation schemes are nontrivial. The two expectation values need not agree: `fdaoe` is designed to preserve a different notion of locality in operator space.


In [ ]:
L = 8
ps = siteinds("Pauli", L)
dt = 0.05
steps = 24
gamma = 0.3
D = daoe(ps, 2, gamma)
FD = fdaoe(ps, 2, gamma)

h2 = spin((1.0, "xx"), (1.0, "yy"), (1.0, "zz"))
superop = commutation_mat(h2)
gates = tebd4(fill(superop, L - 1), ps, dt)

psi = productMPS(ps, ["X", "Z", fill("I", L - 2)...])
times = collect(0:dt:steps * dt)
wd = zeros(length(times))
wf = zeros(length(times))

wd[1] = normalized_weight(D, psi)
wf[1] = normalized_weight(FD, psi)
for n in 1:steps
    psi = apply(gates, psi)
    normalize!(psi)
    wd[n + 1] = normalized_weight(D, psi)
    wf[n + 1] = normalized_weight(FD, psi)
end

(;
    times = times,
    daoe_weight = wd,
    fdaoe_weight = wf,
)


## A minimal filtered evolution loop

This is the direct pattern for an MPS-based truncated operator-growth calculation: evolve the Pauli MPS, apply `fdaoe`, and renormalize. Tracking the bond dimensions gives a quick sanity check that the filter is actually restraining the state complexity.


In [ ]:
bond_dims(psi) = [linkdim(psi, b) for b in 1:length(psi)-1]

psi_raw = productMPS(ps, ["X", "Z", fill("I", L - 2)...])
psi_ftrunc = copy(psi_raw)
raw_dims = Int[]
trunc_dims = Int[]

for _ in 1:steps
    psi_raw = apply(gates, psi_raw)
    normalize!(psi_raw)
    push!(raw_dims, maximum(bond_dims(psi_raw)))

    psi_ftrunc = apply(gates, psi_ftrunc)
    psi_ftrunc = apply(FD, psi_ftrunc)
    normalize!(psi_ftrunc)
    push!(trunc_dims, maximum(bond_dims(psi_ftrunc)))
end

(;
    step = collect(1:steps),
    raw_dims = raw_dims,
    filtered_dims = trunc_dims,
)
